# Gán nhãn ngữ liệu aspect-sentiment cho chủ đề Khách Sạn (Hotel)

Notebook này triển khai bài toán **Gán nhãn ngữ liệu aspect-sentiment cho chủ đề Khách Sạn (Hotel)** cho tiếng Việt theo hướng **multi-label classification**.  
Mỗi câu đầu vào có thể gắn với nhiều cặp **aspect + sentiment** cùng lúc, ví dụ: `room_positive`, `service_negative`.

Quy trình chính của notebook gồm:
1. Đọc dữ liệu từ file `.txt` theo định dạng VLSP.
2. Tách nhãn aspect-sentiment bằng biểu thức chính quy.
3. Tạo vocabulary và mã hoá văn bản thành chuỗi id.
4. Mã hoá nhãn đa nhãn bằng `MultiLabelBinarizer`.
5. Xây dựng `Dataset` và `DataLoader`.
6. Huấn luyện mô hình `BiLSTM`.
7. Đánh giá bằng `micro F1` và `macro F1`.
8. Thử dự đoán trên một câu mẫu.

## Yêu cầu thư viện
Notebook sử dụng `torch`, `numpy`, `scikit-learn` và một số thư viện chuẩn của Python.  
Dữ liệu train/dev/test phải đặt cùng thư mục với notebook.

## Tải dữ liệu lên môi trường
Hai dòng này chỉ dùng trong Google Colab để upload file dữ liệu từ máy tính lên session hiện tại. Nếu chạy trên Jupyter local thì có thể bỏ qua.

In [ ]:
# from google.colab import files
# files.upload()

## Import thư viện và thiết lập cấu hình
Phần này import các thư viện cần thiết, chọn thiết bị chạy (`cuda` nếu có GPU, ngược lại dùng `cpu`) và khai báo các tham số huấn luyện như độ dài tối đa của câu, batch size, số chiều embedding, hidden size, số epoch và learning rate.

In [ ]:
import re
import torch
import numpy as np
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"

MAX_LEN = 100
BATCH_SIZE = 32
EMBED_DIM = 128
HIDDEN_DIM = 128
EPOCHS = 15
LR = 1e-3


## Đọc và chuẩn hoá dữ liệu
Hàm `read_data()` đọc từng file dữ liệu theo định dạng VLSP. Mỗi mẫu gồm một câu và một dòng nhãn. Biểu thức chính quy `\{(.*?),\s*(.*?)\}` được dùng để tách từng cặp `(aspect, sentiment)` rồi ghép thành nhãn dạng `aspect_sentiment`.

In [ ]:
# ===============================
# LOAD DATA
# Format:
# #1
# text...
# LABEL1: positive
# LABEL2: negative
#
# ===============================

def read_data(path):
    samples = []

    with open(path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    i = 0
    while i < len(lines):

        if lines[i].startswith("#"):
            i += 1

            # sentence
            text = lines[i]
            i += 1

            # label line
            label_line = lines[i]
            i += 1

            labels = []

            matches = re.findall(r'\{(.*?),\s*(.*?)\}', label_line)

            for aspect, senti in matches:
                label = aspect.strip() + "_" + senti.strip()
                labels.append(label)

            samples.append((text, labels))

        else:
            i += 1

    return samples


train_data = read_data("1-VLSP2018-SA-Hotel-train (7-3-2018).txt")
dev_data   = read_data("2-VLSP2018-SA-Hotel-dev (7-3-2018).txt")
test_data  = read_data("3-VLSP2018-SA-Hotel-test (8-3-2018).txt")

print("Train:", len(train_data))
print("Dev:", len(dev_data))
print("Test:", len(test_data))


Train: 2999
Dev: 1999
Test: 599


## Tách từ đơn giản
Hàm `tokenize()` chuyển văn bản về chữ thường và tách từ bằng khoảng trắng. Đây là cách tiền xử lý rất cơ bản, phù hợp với bài thử nghiệm đơn giản nhưng chưa xử lý tốt dấu câu hay tiếng Việt phức tạp.

In [ ]:
# ===============================
# TOKENIZER
# ===============================
def tokenize(text):
    return text.lower().split()


## Xây dựng vocabulary
Từ tập train, notebook đếm số lần xuất hiện của từng token bằng `Counter`, rồi gán id cho từng từ. `<pad>` được gán id 0 để đệm chuỗi, `<unk>` được gán id 1 cho các từ không có trong từ điển.

In [ ]:
counter = Counter()

for text, _ in train_data:
    counter.update(tokenize(text))

vocab = {
    "<pad>": 0,
    "<unk>": 1
}

for word, _ in counter.items():
    vocab[word] = len(vocab)

vocab_size = len(vocab)
print("Vocab size:", vocab_size)


Vocab size: 9106


## Mã hoá nhãn đa nhãn
`MultiLabelBinarizer` chuyển mỗi danh sách nhãn thành vector nhị phân. Với mỗi mẫu, vị trí nào có nhãn sẽ mang giá trị 1, các vị trí còn lại là 0. Cách này phù hợp vì một câu có thể chứa nhiều aspect-sentiment cùng lúc.

In [ ]:
mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform([x[1] for x in train_data])
y_dev   = mlb.transform([x[1] for x in dev_data])
y_test  = mlb.transform([x[1] for x in test_data])

num_labels = len(mlb.classes_)
print("Labels:", num_labels)

Labels: 92


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['HOTEL#MISCELLANEOUS_neutral', 'ROOMS#MISCELLANEOUS_positive', 'ROOM_AMENITIES#CLEANLINESS_neutral', 'ROOM_AMENITIES#MISCELLANEOUS_positive', 'ROOM_AMENITIES#PRICES_negative', 'ROOM_AMENITIES#PRICES_positive'] will be ignored
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['HOTEL#MISCELLANEOUS_neutral', 'ROOMS#MISCELLANEOUS_neutral', 'ROOMS#MISCELLANEOUS_positive', 'ROOM_AMENITIES#PRICES_negative'] will be ignored
  warnings.warn(


## Mã hoá văn bản thành chuỗi số
Hàm `encode_text()` đổi từng token sang id trong vocabulary. Chuỗi được cắt hoặc đệm về độ dài cố định `MAX_LEN` để tạo đầu vào có kích thước đồng nhất cho mô hình.

In [ ]:
def encode_text(text):
    ids = []
    for token in tokenize(text):
        ids.append(vocab.get(token, 1))

    ids = ids[:MAX_LEN]

    while len(ids) < MAX_LEN:
        ids.append(0)

    return ids

## Dataset và DataLoader
`ABSADataset` đóng gói dữ liệu thành dạng mà PyTorch có thể dùng trực tiếp. Mỗi phần tử trả về gồm `x` là chuỗi id của câu và `y` là vector nhãn nhị phân. `DataLoader` sau đó chia dữ liệu thành từng batch để train và đánh giá.

In [ ]:
class ABSADataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx][0]
        x = torch.tensor(encode_text(text), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, y


train_ds = ABSADataset(train_data, y_train)
dev_ds   = ABSADataset(dev_data, y_dev)
test_ds  = ABSADataset(test_data, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
dev_loader   = DataLoader(dev_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

## Mô hình BiLSTM
Mô hình gồm 3 phần: `Embedding` để biến token id thành vector, `BiLSTM` để học ngữ cảnh hai chiều, và `Linear` để dự đoán xác suất cho từng nhãn. Sau LSTM, notebook lấy trung bình theo chiều thời gian để gom thông tin toàn câu.

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)

        self.lstm = nn.LSTM(
            EMBED_DIM,
            HIDDEN_DIM,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(HIDDEN_DIM * 2, num_labels)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)

        pooled = torch.mean(out, dim=1)
        logits = self.fc(pooled)
        return logits


model = BiLSTM().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


## Hàm đánh giá
Hàm `evaluate()` chạy mô hình ở chế độ suy luận, áp dụng `sigmoid` để đổi logits sang xác suất, rồi ngưỡng hoá tại 0.5 để lấy nhãn dự đoán. Kết quả được đo bằng `micro F1` và `macro F1`.

In [ ]:
def evaluate(loader):
    model.eval()

    preds = []
    golds = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)

            logits = model(x)
            prob = torch.sigmoid(logits)

            pred = (prob > 0.5).cpu().numpy().astype(np.int32)
            gold = y.cpu().numpy().astype(np.int32)

            preds.append(pred)
            golds.append(gold)

    preds = np.concatenate(preds, axis=0)
    golds = np.concatenate(golds, axis=0)

    print("pred shape:", preds.shape)
    print("gold shape:", golds.shape)
    print("pred dtype:", preds.dtype)
    print("gold dtype:", golds.dtype)

    micro = f1_score(golds, preds, average="micro")
    macro = f1_score(golds, preds, average="macro")

    return micro, macro


## Train
Mỗi epoch, mô hình được train trên tập huấn luyện, sau đó đánh giá trên tập dev. Nếu `dev_micro` tốt hơn mốc tốt nhất trước đó thì notebook lưu trọng số vào file `best_model.pt`.

In [ ]:
best_dev = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    dev_micro, dev_macro = evaluate(dev_loader)

    print(f"Epoch {epoch+1}")
    print("Loss:", round(total_loss, 4))
    print("Dev Micro F1:", round(dev_micro, 4))
    print("Dev Macro F1:", round(dev_macro, 4))
    print("-"*40)

    if dev_micro > best_dev:
        best_dev = dev_micro
        torch.save(model.state_dict(), "best_model.pt")



pred shape: (1999, 92)
gold shape: (1999, 92)
pred dtype: int32
gold dtype: int32
Epoch 1
Loss: 21.7369
Dev Micro F1: 0.2446
Dev Macro F1: 0.0078
----------------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


pred shape: (1999, 92)
gold shape: (1999, 92)
pred dtype: int32
gold dtype: int32
Epoch 2
Loss: 14.2147
Dev Micro F1: 0.2227
Dev Macro F1: 0.0084
----------------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


pred shape: (1999, 92)
gold shape: (1999, 92)
pred dtype: int32
gold dtype: int32
Epoch 3
Loss: 13.9743
Dev Micro F1: 0.1906
Dev Macro F1: 0.0083
----------------------------------------


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


pred shape: (1999, 92)
gold shape: (1999, 92)
pred dtype: int32
gold dtype: int32
Epoch 14
Loss: 9.6815
Dev Micro F1: 0.4373
Dev Macro F1: 0.0615
----------------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


pred shape: (1999, 92)
gold shape: (1999, 92)
pred dtype: int32
gold dtype: int32
Epoch 15
Loss: 9.3857
Dev Micro F1: 0.4475
Dev Macro F1: 0.0643
----------------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Đánh giá trên tập test
Sau khi train xong, notebook nạp lại checkpoint tốt nhất và tính chỉ số trên tập test. Đây là kết quả cuối cùng để báo cáo chất lượng mô hình.

In [ ]:
model.load_state_dict(torch.load("best_model.pt"))

test_micro, test_macro = evaluate(test_loader)

print("Final Test")
print("Micro F1:", round(test_micro, 4))
print("Macro F1:", round(test_macro, 4))


pred shape: (599, 92)
gold shape: (599, 92)
pred dtype: int32
gold dtype: int32
Final Test
Micro F1: 0.4404
Macro F1: 0.0687


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Thử dự đoán một câu mẫu
Hàm `predict()` nhận một câu mới, mã hoá câu đó, chạy mô hình và in ra các nhãn có xác suất lớn hơn 0.5. Phần này giúp kiểm tra nhanh mô hình hoạt động ra sao trên dữ liệu mới.

In [ ]:
def predict(text):
    model.eval()

    x = torch.tensor([encode_text(text)], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(x)
        prob = torch.sigmoid(logits)[0].cpu().numpy()

    for i, p in enumerate(prob):
        if p > 0.5:
            print(mlb.classes_[i], round(float(p), 3))


predict("Khách sạn gần biển, giá rẻ")

LOCATION#GENERAL_positive 0.955
